# Select a Tutorial

Pick a tutorial below. Clicking **Open tutorial** will:

1. download (`git clone`) the tutorial into `~/tutorials/<name>` (skipped if it's already there),
2. switch the file browser on the left to the tutorial's folder, and
3. open the tutorial's main notebook in a new tab.

> The tiles appear automatically a few seconds after this page opens. If they don't, use **Run ▸ Run All Cells** (the code that builds them is collapsed below — click the ⋯ bar to see it).


In [ ]:
import json
import os
import subprocess
from pathlib import Path

import yaml
import ipywidgets as widgets
from IPython.display import HTML, Javascript, display

# --------------------------------------------------------------------------
# The tutorials shown as tiles are defined in build/tutorials.yaml.
# To add or change a tutorial, edit that file (no need to touch this code).
# Each entry has: title, emoji, description, repo (git clone URL), notebook.
# --------------------------------------------------------------------------
def _find_tutorials_yaml():
    candidates = [
        Path("build") / "tutorials.yaml",                       # kernel started in recipe-demos/
        Path.home() / "welcome" / "tutorials.yaml",
        Path.cwd() / "tutorials.yaml",
    ]
    for path in candidates:
        if path.is_file():
            return path
    raise FileNotFoundError(
        "Could not find build/tutorials.yaml. Looked in:\n  "
        + "\n  ".join(str(c) for c in candidates)
    )


with open(_find_tutorials_yaml()) as f:
    TUTORIALS = yaml.safe_load(f)

# Where tutorials are cloned (one sub-folder per repo).
TUTORIALS_DIR = Path.home() / "tutorials"
# Root of JupyterLab's file browser; paths handed to JupyterLab are relative to it.
SERVER_ROOT = Path(os.environ.get("JUPYTER_SERVER_ROOT", Path.home())).resolve()


def repo_dir(tut):
    name = tut["repo"].rstrip("/").rsplit("/", 1)[-1].removesuffix(".git")
    return TUTORIALS_DIR / name


def open_in_jupyterlab(path):
    """Tell JupyterLab to cd the file browser to `path`'s folder and open it."""
    lab_path = os.path.relpath(Path(path).resolve(), SERVER_ROOT)
    args = json.dumps({"path": lab_path})
    # JupyterLab runs a command when any element carrying these attributes is clicked.
    button = (
        f'<button data-commandlinker-command="filebrowser:open-path" '
        f"data-commandlinker-args='{args}'>Open {Path(path).name}</button>"
    )
    display(HTML(f"Opening <b>{lab_path}</b> … (if it did not open: {button})"))
    display(
        Javascript(
            """
            (function () {
              var el = document.createElement('span');
              el.setAttribute('data-commandlinker-command', 'filebrowser:open-path');
              el.setAttribute('data-commandlinker-args', %s);
              document.body.appendChild(el);
              el.click();
              el.remove();
            })();
            """
            % json.dumps(args)
        )
    )


def launch(tut):
    dest = repo_dir(tut)
    if dest.exists():
        print(f"✓ {tut['title']} is already downloaded at {dest}")
    else:
        TUTORIALS_DIR.mkdir(parents=True, exist_ok=True)
        print(f"⬇ Downloading {tut['repo']} → {dest} …", flush=True)
        result = subprocess.run(
            ["git", "clone", tut["repo"], str(dest)], capture_output=True, text=True
        )
        if result.returncode != 0:
            print("✗ git clone failed:\n" + result.stderr)
            return
        print("✓ Download complete.")
    notebook = dest / tut["notebook"]
    if not notebook.exists():
        print(f"✗ Could not find {notebook} — open the folder {dest} and pick a notebook.")
        open_in_jupyterlab(dest)
        return
    open_in_jupyterlab(notebook)


# ---------------------------------- UI -----------------------------------
status = widgets.Output()
buttons = []


def on_click(button, tut):
    status.clear_output()
    for b in buttons:
        b.disabled = True
    button.description = "Working …"
    try:
        with status:
            launch(tut)
    finally:
        button.description = "Open tutorial"
        for b in buttons:
            b.disabled = False


def make_tile(tut):
    card = widgets.HTML(
        f"""
        <div style="text-align:center; padding:10px 8px 2px 8px;">
          <div style="font-size:40px; line-height:1.1;">{tut['emoji']}</div>
          <div style="font-weight:600; font-size:15px; margin-top:8px;">{tut['title']}</div>
          <div style="font-size:12px; color:var(--jp-ui-font-color2); margin-top:6px; min-height:54px;">
            {tut['description']}
          </div>
        </div>"""
    )
    button = widgets.Button(
        description="Open tutorial",
        button_style="primary",
        tooltip=tut["repo"],
        layout=widgets.Layout(width="auto", margin="6px 12px 12px 12px"),
    )
    button.on_click(lambda b, tut=tut: on_click(b, tut))
    buttons.append(button)
    tile = widgets.VBox(
        [card, button],
        layout=widgets.Layout(
            width="220px",
            margin="8px",
            border="1px solid var(--jp-border-color2)",
            align_items="stretch",
        ),
    )
    tile.add_class("tutorial-tile")
    return tile


display(
    HTML(
        """<style>
        .tutorial-tile { border-radius: 10px; background: var(--jp-layout-color1); }
        .tutorial-tile:hover { box-shadow: 0 2px 10px rgba(0,0,0,.18); }
        </style>"""
    )
)
display(widgets.HBox([make_tile(t) for t in TUTORIALS], layout=widgets.Layout(flex_flow="row wrap")))
display(status)


---
### What happens when you open a tutorial

- Click **Open tutorial** on any card above. The tutorial is downloaded into your `tutorials/` folder and opens automatically in a new tab — just follow along from there.
- Already opened one before? It won't download again; it simply reopens.
- **Come back any time:** reopen this page from the **Select a Tutorial** tile on the Launcher (open a new Launcher with the **+** button above the file browser on the left).
